- Update: 2023.07.04

# 0. 配置参数

- `source`：待查重数据源路径
- `out`：保存结果的路径

In [1]:
source = r'C:\Users\YangTeng\Desktop\文档\查重PPT代码和练习\Asher【模板】.xlsx'# 输入带查重文件路径

out = r'C:\Users\YangTeng\Desktop\文档\查重PPT代码和练习\【查重结果】刹车制动钳目录_NOW().xlsx' # 输入输出文件路径

In [2]:
import pandas as pd

In [3]:
# 读取待查重数据
fac_data = pd.read_excel(source,sheet_name='需查重数据',dtype=str).fillna('')

fac_data

,序号,辨识号,OE
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...
1,2,YT002,5E090103;883102B720;883102B721
2,3,YT003,5E120505;8831005080;8831068010
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...
...,...,...,...
147,148,YT148,PX167171;LR035760
148,149,YT149,PX167373;36001670;36011357
149,150,YT150,PX167575;36000283
150,151,YT151,PX167878;31332528


## 定义清理OE函数

In [4]:
import re

In [5]:
def normalise_oe(oe):
    
    oe = str(oe).strip()
#     oe = re.sub(r'^[a-zA-Z]*:', '', oe)#去除":"及前面所有字母
    oe = re.sub(r'[\u4e00-\u9fa5]',"",oe) #去除汉字
    oe = re.sub('\.+','',oe)  #去除"."符号
    oe = re.sub('\（','(',oe) #将所有中文"（"符号替换为英文"("
    oe = re.sub('\）',')',oe) #将所有中文"）"符号替换为英文")"
    oe = re.sub('\(.*\)','',oe)  #删除所有"()"及其内部的字符（因为通常不是OE号）
    oe = re.sub('\*+', '', oe) #删除所有的"*"
    oe = re.sub('  +', ';', oe) #将两个以上的空格替换为';'
    oe = re.sub(' ', '', oe) #删除单空格
    oe = re.sub('/+', ';', oe) #将“/"替换为';'
    oe = re.sub('\n', ';', oe) #将换行符替换为';'
    oe = re.sub('[；，,+]', ';', oe) #将[；，,+]替换为';'
    oe = re.sub('\u200e', '', oe)
    oe = re.sub(';$','',oe)
    oe = re.sub('\-', '', oe)

    #list1 = oe26.split(';')
    list1 = [i for i in str(oe).split(';') if i != '' and i.isalpha() == False] #去除空格
    list1 = [re.sub(r'[^0-9]+[:：]','',i) for i in list1]#去除":"及前面所有字母
    list1 = [re.sub(r'[:：]+','',i) for i in list1]#去除":"
    list1 = [i for i in list1 if i !=''and i.isalpha() == False] #去除空格
    
    list2 = list(set(list1))
    list2.sort(key = list1.index)# 去重后保持OE号原来的顺序
    
    oe=';'.join(list2)
    
# #     删除OE号中混杂的年份如'1999';'2018'
    
#     list3 = [i for i in str(oe27).split(';') if len(i) > 3]
    
#     list3 = [i for i in list3 if not (len(i) == 4 and i[0] in ['1', '2'])]

    
#     oe27=';'.join(list3)
    
    return oe.upper()

## 内部查重

In [6]:
fac_data['清理后OE'] = [normalise_oe(fac_data.loc[i, 'OE']) for i in range(len(fac_data))]

fac_data

,序号,辨识号,OE,清理后OE
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...
...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357
149,150,YT150,PX167575;36000283,PX167575;36000283
150,151,YT151,PX167878;31332528,PX167878;31332528


In [7]:
# 剔除无OE的数据
clean_fac_data = fac_data[fac_data['清理后OE']!='']

clean_fac_data

,序号,辨识号,OE,清理后OE
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...
...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357
149,150,YT150,PX167575;36000283,PX167575;36000283
150,151,YT151,PX167878;31332528,PX167878;31332528


In [8]:
new_fac_data = clean_fac_data[['辨识号','清理后OE']].copy()
new_fac_data['清理后OE'] = new_fac_data['清理后OE'].str.split(';')
new_fac_data = new_fac_data.explode('清理后OE')

new_fac_data

,辨识号,清理后OE
0,YT001,5E090102
0,YT001,8831002470
0,YT001,8831002630
0,YT001,8831002640
0,YT001,883100D200
...,...,...
149,YT150,36000283
150,YT151,PX167878
150,YT151,31332528
151,YT152,VS164039


In [9]:
merge_fac_data = pd.merge(new_fac_data,new_fac_data,on='清理后OE',how='left')

merge_fac_data

,辨识号_x,清理后OE,辨识号_y
0,YT001,5E090102,YT001
1,YT001,8831002470,YT001
2,YT001,8831002630,YT001
3,YT001,8831002640,YT001
4,YT001,883100D200,YT001
...,...,...,...
1093,YT150,36000283,YT150
1094,YT151,PX167878,YT151
1095,YT151,31332528,YT151
1096,YT152,VS164039,YT152


In [10]:
group_merge_fac_data = merge_fac_data.groupby(['清理后OE'],as_index=False,sort=False).agg({'辨识号_y':';'.join})

group_merge_fac_data

,清理后OE,辨识号_y
0,5E090102,YT001
1,8831002470,YT001
2,8831002630,YT001
3,8831002640,YT001
4,883100D200,YT001
...,...,...
939,36000283,YT150
940,PX167878,YT151
941,31332528,YT151
942,VS164039,YT152


In [11]:
repeat_fac_data = pd.merge(new_fac_data,group_merge_fac_data,on='清理后OE',how='left').fillna('')

repeat_fac_data

,辨识号,清理后OE,辨识号_y
0,YT001,5E090102,YT001
1,YT001,8831002470,YT001
2,YT001,8831002630,YT001
3,YT001,8831002640,YT001
4,YT001,883100D200,YT001
...,...,...,...
979,YT150,36000283,YT150
980,YT151,PX167878,YT151
981,YT151,31332528,YT151
982,YT152,VS164039,YT152


In [12]:
fac_repeat_data = repeat_fac_data[repeat_fac_data['辨识号']!=repeat_fac_data['辨识号_y']]

fac_repeat_data

,辨识号,清理后OE,辨识号_y
17,YT005,1K0820803E,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
18,YT005,1K0820803G,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
19,YT005,1K0820803P,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
20,YT005,1K0820803Q,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
21,YT005,1K0820803S,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
...,...,...,...
583,YT058,A0002308511,YT055;YT058;YT055;YT058
714,YT075,64509174803,YT075;YT132;YT075;YT132
930,YT128,64526901781,YT128;YT131;YT128;YT131
940,YT131,64526901781,YT128;YT131;YT128;YT131


In [13]:
group_fac_repeat_data = fac_repeat_data.groupby(['辨识号'],as_index=False,sort=False).agg({'清理后OE':';'.join, '辨识号_y':';'.join})

group_fac_repeat_data

,辨识号,清理后OE,辨识号_y
0,YT005,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
1,YT006,8E0260805BS,YT006;YT010;YT006;YT010
2,YT007,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
3,YT010,8E0260805BS,YT006;YT010;YT006;YT010
4,YT017,4472603500,YT017;YT018;YT017;YT018
5,YT018,4472603500,YT017;YT018;YT017;YT018
6,YT025,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
7,YT045,2483000770,YT045;YT047;YT045;YT047
8,YT047,2483000770,YT045;YT047;YT045;YT047
9,YT055,A0002308511,YT055;YT058;YT055;YT058


In [14]:
group_fac_repeat_data.rename(columns={'清理后OE':'表内重复OE号','辨识号_y':'表内重复辨识号'},inplace=True)

In [15]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号
0,YT005,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
1,YT006,8E0260805BS,YT006;YT010;YT006;YT010
2,YT007,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
3,YT010,8E0260805BS,YT006;YT010;YT006;YT010
4,YT017,4472603500,YT017;YT018;YT017;YT018
5,YT018,4472603500,YT017;YT018;YT017;YT018
6,YT025,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007;YT025;YT005;YT007;YT025;YT005;YT00...
7,YT045,2483000770,YT045;YT047;YT045;YT047
8,YT047,2483000770,YT045;YT047;YT045;YT047
9,YT055,A0002308511,YT055;YT058;YT055;YT058


In [16]:
group_fac_repeat_data['表内重复OE号'] = group_fac_repeat_data['表内重复OE号'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))
group_fac_repeat_data['表内重复辨识号'] = group_fac_repeat_data['表内重复辨识号'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))

In [17]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号
0,YT005,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT005, YT007, YT025]"
1,YT006,[8E0260805BS],"[YT006, YT010]"
2,YT007,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT005, YT007, YT025]"
3,YT010,[8E0260805BS],"[YT006, YT010]"
4,YT017,[4472603500],"[YT017, YT018]"
5,YT018,[4472603500],"[YT017, YT018]"
6,YT025,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT005, YT007, YT025]"
7,YT045,[2483000770],"[YT045, YT047]"
8,YT047,[2483000770],"[YT045, YT047]"
9,YT055,[A0002308511],"[YT055, YT058]"


In [18]:
for i in range(len(group_fac_repeat_data)):
    for item in group_fac_repeat_data['表内重复辨识号'][i]:
        if item == group_fac_repeat_data['辨识号'][i]:
            print(i)
            print(group_fac_repeat_data['辨识号'][i])
            print(item)
            group_fac_repeat_data['表内重复辨识号'][i].remove(item)

0
YT005
YT005
1
YT006
YT006
2
YT007
YT007
3
YT010
YT010
4
YT017
YT017
5
YT018
YT018
6
YT025
YT025
7
YT045
YT045
8
YT047
YT047
9
YT055
YT055
10
YT058
YT058
11
YT075
YT075
12
YT128
YT128
13
YT131
YT131
14
YT132
YT132


In [19]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号
0,YT005,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT007, YT025]"
1,YT006,[8E0260805BS],[YT010]
2,YT007,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT005, YT025]"
3,YT010,[8E0260805BS],[YT006]
4,YT017,[4472603500],[YT018]
5,YT018,[4472603500],[YT017]
6,YT025,"[1K0820803E, 1K0820803G, 1K0820803P, 1K0820803...","[YT005, YT007]"
7,YT045,[2483000770],[YT047]
8,YT047,[2483000770],[YT045]
9,YT055,[A0002308511],[YT058]


In [20]:
group_fac_repeat_data['表内重复OE号'] = group_fac_repeat_data['表内重复OE号'].apply(lambda x: ';'.join(x))
group_fac_repeat_data['表内重复辨识号'] = group_fac_repeat_data['表内重复辨识号'].apply(lambda x: ';'.join(x))

In [21]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号
0,YT005,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT007;YT025
1,YT006,8E0260805BS,YT010
2,YT007,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT025
3,YT010,8E0260805BS,YT006
4,YT017,4472603500,YT018
5,YT018,4472603500,YT017
6,YT025,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT005;YT007
7,YT045,2483000770,YT047
8,YT047,2483000770,YT045
9,YT055,A0002308511,YT058


In [22]:
result = pd.merge(fac_data,group_fac_repeat_data,on='辨识号',how='left').fillna('')

result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...,,
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721,,
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010,,
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...,,
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT007;YT025
...,...,...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760,,
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357,,
149,150,YT150,PX167575;36000283,PX167575;36000283,,
150,151,YT151,PX167878;31332528,PX167878;31332528,,


In [23]:
result[result['清理后OE']=='']

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号


## 和PTSKU查重

In [24]:
# 读取系统数据
sku_data = pd.read_excel(source,sheet_name='PTSKU',dtype=str).fillna('')

sku_data

,SKU,OE
0,ST350707,ST350707;CO10968C;1K2A661450;1K2A661450A;1K2A6...
1,ST770306,ST770306;CO105116C;78396;77396;100119;64526904...
2,ST780116,ST780116;CO10954C;158306;977013K425;977013K700
3,ST960618,ST960618;CO11440C;157322;168318;4M0820803;4M08...
4,ST100311A0,ST100311A0;CO101230C;57123;F3AZ19V703AA
...,...,...
1466,15002061,15002061;5412301011;A5412301011
1467,77504013,77504013;5412300611;5412301211;A5412300611;A54...
1468,76209209,76209209;36000271;8603392;8603685
1469,66001003,66001003;36000270;8603230;8603684


In [25]:
sku_data['清理后OE'] = [normalise_oe(sku_data.loc[i, 'OE']) for i in range(len(sku_data))]

sku_data

,SKU,OE,清理后OE
0,ST350707,ST350707;CO10968C;1K2A661450;1K2A661450A;1K2A6...,ST350707;CO10968C;1K2A661450;1K2A661450A;1K2A6...
1,ST770306,ST770306;CO105116C;78396;77396;100119;64526904...,ST770306;CO105116C;78396;77396;100119;64526904...
2,ST780116,ST780116;CO10954C;158306;977013K425;977013K700,ST780116;CO10954C;158306;977013K425;977013K700
3,ST960618,ST960618;CO11440C;157322;168318;4M0820803;4M08...,ST960618;CO11440C;157322;168318;4M0820803;4M08...
4,ST100311A0,ST100311A0;CO101230C;57123;F3AZ19V703AA,ST100311A0;CO101230C;57123;F3AZ19V703AA
...,...,...,...
1466,15002061,15002061;5412301011;A5412301011,15002061;5412301011;A5412301011
1467,77504013,77504013;5412300611;5412301211;A5412300611;A54...,77504013;5412300611;5412301211;A5412300611;A54...
1468,76209209,76209209;36000271;8603392;8603685,76209209;36000271;8603392;8603685
1469,66001003,66001003;36000270;8603230;8603684,66001003;36000270;8603230;8603684


In [26]:
clean_sku_data = sku_data[sku_data['清理后OE']!='']

clean_sku_data

,SKU,OE,清理后OE
0,ST350707,ST350707;CO10968C;1K2A661450;1K2A661450A;1K2A6...,ST350707;CO10968C;1K2A661450;1K2A661450A;1K2A6...
1,ST770306,ST770306;CO105116C;78396;77396;100119;64526904...,ST770306;CO105116C;78396;77396;100119;64526904...
2,ST780116,ST780116;CO10954C;158306;977013K425;977013K700,ST780116;CO10954C;158306;977013K425;977013K700
3,ST960618,ST960618;CO11440C;157322;168318;4M0820803;4M08...,ST960618;CO11440C;157322;168318;4M0820803;4M08...
4,ST100311A0,ST100311A0;CO101230C;57123;F3AZ19V703AA,ST100311A0;CO101230C;57123;F3AZ19V703AA
...,...,...,...
1466,15002061,15002061;5412301011;A5412301011,15002061;5412301011;A5412301011
1467,77504013,77504013;5412300611;5412301211;A5412300611;A54...,77504013;5412300611;5412301211;A5412300611;A54...
1468,76209209,76209209;36000271;8603392;8603685,76209209;36000271;8603392;8603685
1469,66001003,66001003;36000270;8603230;8603684,66001003;36000270;8603230;8603684


In [27]:
new_sku_data = clean_sku_data[['SKU','清理后OE']].copy()
new_sku_data['清理后OE'] = new_sku_data['清理后OE'].str.split(';')
new_sku_data = new_sku_data.explode('清理后OE')

new_sku_data

,SKU,清理后OE
0,ST350707,ST350707
0,ST350707,CO10968C
0,ST350707,1K2A661450
0,ST350707,1K2A661450A
0,ST350707,1K2A661L10A
...,...,...
1469,66001003,8603230
1469,66001003,8603684
1470,30011011,30011011
1470,30011011,4472800055


In [28]:
merge_data = pd.merge(new_fac_data,new_sku_data,on='清理后OE',how='left').fillna('')

merge_data

,辨识号,清理后OE,SKU
0,YT001,5E090102,
1,YT001,8831002470,
2,YT001,8831002630,
3,YT001,8831002640,
4,YT001,883100D200,59004002
...,...,...,...
1254,YT150,36000283,
1255,YT151,PX167878,
1256,YT151,31332528,
1257,YT152,VS164039,


In [29]:
repeat_data = merge_data[merge_data['SKU']!='']

repeat_data

,辨识号,清理后OE,SKU
4,YT001,883100D200,59004002
5,YT001,883100D200,59004002
10,YT003,8831005080,ST531005
11,YT003,8831005080,ST531005
12,YT003,8831068010,ST531005
...,...,...,...
1241,YT145,6854078,64002025
1245,YT147,1K0820808H,ST794464
1246,YT147,1K0820808H,ST794464
1251,YT149,36011357,78081311


In [30]:
sku_repeat_data = repeat_data.groupby(['辨识号'],as_index=False,sort=False).agg({'清理后OE':';'.join,'SKU':';'.join})

sku_repeat_data

,辨识号,清理后OE,SKU
0,YT001,883100D200;883100D200,59004002;59004002
1,YT003,8831005080;8831005080;8831068010;8831068010,ST531005;ST531005;ST531005;ST531005
2,YT004,8831005101;8831005101;883100F020;883100F020;88...,52012011;52012011;52012011;52012011;52012011;5...
3,YT005,7E170101;1K0820803E;1K0820803E;1K0820803E;1K08...,7E170101;7E170101;78203003;78203003;BL.07.86;B...
4,YT006,6E140203;4F0260805P;4F0260805P;8E0260805BA;8E0...,6E140203;6E140203;BL.06.37;ST960103GW;68005011...
...,...,...,...
92,YT139,4471808381;4471808381,ST795270;ST795270
93,YT144,64509121760,ST810505
94,YT145,6854078;6854078,ST680204;64002025
95,YT147,1K0820808H;1K0820808H,ST794464;ST794464


In [31]:
sku_repeat_data.rename(columns={'清理后OE':'系统重复OE','SKU':'重复PTSKU'},inplace=True)

sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,YT001,883100D200;883100D200,59004002;59004002
1,YT003,8831005080;8831005080;8831068010;8831068010,ST531005;ST531005;ST531005;ST531005
2,YT004,8831005101;8831005101;883100F020;883100F020;88...,52012011;52012011;52012011;52012011;52012011;5...
3,YT005,7E170101;1K0820803E;1K0820803E;1K0820803E;1K08...,7E170101;7E170101;78203003;78203003;BL.07.86;B...
4,YT006,6E140203;4F0260805P;4F0260805P;8E0260805BA;8E0...,6E140203;6E140203;BL.06.37;ST960103GW;68005011...
...,...,...,...
92,YT139,4471808381;4471808381,ST795270;ST795270
93,YT144,64509121760,ST810505
94,YT145,6854078;6854078,ST680204;64002025
95,YT147,1K0820808H;1K0820808H,ST794464;ST794464


In [32]:
sku_repeat_data['系统重复OE'] = sku_repeat_data['系统重复OE'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))
sku_repeat_data['重复PTSKU'] = sku_repeat_data['重复PTSKU'].str.split(';').apply(lambda x: sorted(list((set(x))), key=x.index))

In [33]:
sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,YT001,[883100D200],[59004002]
1,YT003,"[8831005080, 8831068010]",[ST531005]
2,YT004,"[8831005101, 883100F020, 883100F021]",[52012011]
3,YT005,"[7E170101, 1K0820803E, 1K0820803G, 1K0820859S]","[7E170101, 78203003, BL.07.86, BL.07.36]"
4,YT006,"[6E140203, 4F0260805P, 8E0260805BA, 4F0260805A...","[6E140203, BL.06.37, ST960103GW, 68005011]"
...,...,...,...
92,YT139,[4471808381],[ST795270]
93,YT144,[64509121760],[ST810505]
94,YT145,[6854078],"[ST680204, 64002025]"
95,YT147,[1K0820808H],[ST794464]


In [34]:
merge_data

,辨识号,清理后OE,SKU
0,YT001,5E090102,
1,YT001,8831002470,
2,YT001,8831002630,
3,YT001,8831002640,
4,YT001,883100D200,59004002
...,...,...,...
1254,YT150,36000283,
1255,YT151,PX167878,
1256,YT151,31332528,
1257,YT152,VS164039,


In [35]:
sku_repeat_data['系统重复OE'] = sku_repeat_data['系统重复OE'].apply(lambda x: ';'.join(x))
sku_repeat_data['重复PTSKU'] = sku_repeat_data['重复PTSKU'].apply(lambda x: ';'.join(x))

In [36]:
sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,YT001,883100D200,59004002
1,YT003,8831005080;8831068010,ST531005
2,YT004,8831005101;883100F020;883100F021,52012011
3,YT005,7E170101;1K0820803E;1K0820803G;1K0820859S,7E170101;78203003;BL.07.86;BL.07.36
4,YT006,6E140203;4F0260805P;8E0260805BA;4F0260805AC;4F...,6E140203;BL.06.37;ST960103GW;68005011
...,...,...,...
92,YT139,4471808381,ST795270
93,YT144,64509121760,ST810505
94,YT145,6854078,ST680204;64002025
95,YT147,1K0820808H,ST794464


In [37]:
result = pd.merge(result,sku_repeat_data,on='辨识号',how='left').fillna('')

result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...,,,883100D200,59004002
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721,,,,
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010,,,8831005080;8831068010,ST531005
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...,,,8831005101;883100F020;883100F021,52012011
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT007;YT025,7E170101;1K0820803E;1K0820803G;1K0820859S,7E170101;78203003;BL.07.86;BL.07.36
...,...,...,...,...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760,,,,
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357,,,36011357,78081311
149,150,YT150,PX167575;36000283,PX167575;36000283,,,,
150,151,YT151,PX167878;31332528,PX167878;31332528,,,,


In [38]:
result['是否已开发']=''

In [39]:
result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...,,,883100D200,59004002,
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721,,,,,
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010,,,8831005080;8831068010,ST531005,
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...,,,8831005101;883100F020;883100F021,52012011,
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT007;YT025,7E170101;1K0820803E;1K0820803G;1K0820859S,7E170101;78203003;BL.07.86;BL.07.36,
...,...,...,...,...,...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760,,,,,
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357,,,36011357,78081311,
149,150,YT150,PX167575;36000283,PX167575;36000283,,,,,
150,151,YT151,PX167878;31332528,PX167878;31332528,,,,,


In [40]:
for i in range(len(result)):
    if result.loc[i,'清理后OE'] != '':
        if result.loc[i,'重复PTSKU']!='':
            result.loc[i,'是否已开发'] = '是'
        else:
            if result.loc[i,'表内重复辨识号']!='':
                result.loc[i,'是否已开发'] = '否，表内重复'
            else:
                result.loc[i,'是否已开发'] = '否'
    else:
        result.loc[i,'是否已开发'] = '无OE号，待进一步确认'

In [41]:
result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发
0,1,YT001,5E090102;8831002470;8831002630;8831002640;8831...,5E090102;8831002470;8831002630;8831002640;8831...,,,883100D200,59004002,是
1,2,YT002,5E090103;883102B720;883102B721,5E090103;883102B720;883102B721,,,,,否
2,3,YT003,5E120505;8831005080;8831068010,5E120505;8831005080;8831068010,,,8831005080;8831068010,ST531005,是
3,4,YT004,5E120506;8831005100;8831005101;883100F020;8831...,5E120506;8831005100;8831005101;883100F020;8831...,,,8831005101;883100F020;883100F021,52012011,是
4,5,YT005,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,7E170101;1K0820803E;1K0820803G;1K0820803P;1K08...,1K0820803E;1K0820803G;1K0820803P;1K0820803Q;1K...,YT007;YT025,7E170101;1K0820803E;1K0820803G;1K0820859S,7E170101;78203003;BL.07.86;BL.07.36,是
...,...,...,...,...,...,...,...,...,...
147,148,YT148,PX167171;LR035760,PX167171;LR035760,,,,,否
148,149,YT149,PX167373;36001670;36011357,PX167373;36001670;36011357,,,36011357,78081311,是
149,150,YT150,PX167575;36000283,PX167575;36000283,,,,,否
150,151,YT151,PX167878;31332528,PX167878;31332528,,,,,否


In [42]:
result[result['是否已开发']=='无OE号，待进一步确认']

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发


In [43]:
import datetime

In [44]:
result.to_excel(out.replace('NOW()', datetime.datetime.now().strftime("%Y%m%d_%H%M%S")), index=False)